# 06 - Comparison with published frame-selection methods (AKS, Q-Frame)

Runs two published **training-free** selection methods on our subset, against uniform sampling and our two-tier
method, at **8 / 16 / 32 frame budgets**. Nothing in notebooks 01-05 is modified: the shared harness lives in
`asvlm_core.py` (extracted from notebook 05) and every method answers through the **same** Qwen3-VL-4B, the same
official prompts and the same letter-probability scoring.

| Method | Paper | Selection rule | Scorer here |
|---|---|---|---|
| **Uniform** | - | evenly spaced frames | - |
| **AKS** | Adaptive Keyframe Sampling, CVPR 2025 ([code](https://github.com/ncTimTang/AKS)) | recursive binary split on prompt relevance, then top-k inside each segment; balances relevance and coverage | BLIP-ITM (`Salesforce/blip-itm-base-coco`), as in the paper |
| **Q-Frame (fixed frames)** | Q-Frame, ICCV 2025 ([code](https://github.com/xiaomi-research/q-frame)) | Gumbel-Max sampling over `softmax(I / tau)` relevance, without replacement | CLIP ViT-L/14 |
| **Q-Frame (multi-resolution)** | same | same ranking, then high / medium / low resolution tiers at a fixed token budget | CLIP ViT-L/14 |
| **BOLT** | BOLT, CVPR 2025 ([code](https://github.com/sming256/BOLT)) | inverse transform sampling: build a CDF over sharpened relevance scores and sample at even quantiles | CLIP ViT-L/14 |
| **MaxInfo** | MaxInfo, WACV 2026 ([code](https://github.com/FusionBrainLab/MaxInfo)) | maximum-volume subset: frames whose embeddings span the largest volume (query-free diversity) | SigLIP2 embeddings |
| **Ours (two-tier)** | this work | K low-res cluster representatives + high-res close-ups, at the same token budget | SigLIP2 (already computed) |

### Two settings
1. **Fixed frames** (8 / 16 / 32 frames, all at 640x360): uniform, AKS, Q-Frame, BOLT, MaxInfo.
2. **Fixed tokens** (the same budget in visual tokens: 1.8k / 3.5k / 7.0k): Q-Frame multi-resolution and ours,
   which spend the budget across resolutions.

### Faithfulness notes (worth repeating in the paper)
- **Long-CLIP replaced by CLIP ViT-L/14.** Q-Frame uses Long-CLIP for prompts longer than 77 tokens; our
  questions are short, so the long-context variant is not needed.
- **Candidate pool.** Both papers first sample 128 candidate frames. We score the **entire** extracted pool
  (5 fps MVBench, 1 fps Video-MME; 194 frames per question on average), which is the same or denser.
- **Q-Frame's 1/16 resolution tier cannot be reproduced exactly.** Qwen3-VL floors every image at 256x256 pixels
  (~64 visual tokens), so the cheapest tier costs ~64 tokens, not 220/16. We therefore keep Q-Frame's *frame
  ratio* 1 : 2 : 8 (high : medium : low) and scale it to the token budget. At the 16-frame budget this reproduces
  their published 4 + 8 + 32 configuration exactly.
- All methods are reimplemented from their released code / paper formulas; the AKS recursion follows
  `frame_select.py` (`t1 = 0.8`, `t2 = -100`, `all_depth = 5`) line by line.
- **MaxInfo** is query-free by design (pure embedding diversity), so it doubles as the cleanest control for our
  K-means clustering. It runs on the SigLIP2 embeddings we already have; the paper uses the host VLM's encoder.
- **BOLT** uses its inverse-transform-sampling variant (the paper's headline strategy) with the sharpening
  exponent `alpha = 3.0`, inside their reported 2.5-3.5 range.

In [ ]:
import json, math, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
import asvlm_core as core          # shared harness (data loading, prompts, model wrapper, runner)

print("core from", core.__file__, "| cuda", torch.cuda.is_available())

## 1. Configuration

In [ ]:
RESULTS_DIR = core.PROJECT_ROOT / "results" / "papers"
SCORE_DIR = core.FEATURES_ROOT / "paper_scores"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BUDGETS = [8, 16, 32]                     # frames at 640x360; the token budget is 220 x this
LIMIT_PER_CATEGORY = None                 # small int for a smoke test

BLIP_ID = "Salesforce/blip-itm-base-coco" # AKS scorer (the paper's default)
CLIP_ID = "openai/clip-vit-large-patch14" # Q-Frame scorer (stands in for Long-CLIP)

AKS_T1, AKS_T2, AKS_DEPTH = 0.8, -100, 5  # defaults from the AKS repo
QFRAME_TAU = 0.8                          # Q-Frame temperature (paper's ablation optimum)
QFRAME_TIERS = (1, 2, 8)                  # high : medium : low frame ratio (their 4 + 8 + 32)
BOLT_ALPHA = 3.0                          # BOLT score sharpening exponent (paper: 2.5 - 3.5)
MAXINFO_PCA_DIM = 64                      # MaxInfo "fast" variant: reduce dimensions before max-volume selection
OURS_SUMMARY_SHARE = 0.6                  # share of the token budget our method spends on the low-res summary

TEST = core.load_samples("test", limit=LIMIT_PER_CATEGORY)
BY_QID = {s["qid"]: s for s in TEST}
print(len(TEST), "test questions |", sum(len(s["frames"]) for s in TEST), "candidate frames in total")

## 2. Relevance scorers

Both scorers run once per question over that question's whole frame pool and are cached under
`data/features/paper_scores/`, so re-running costs nothing.

- **BLIP-ITM (AKS):** the image-text matching head, i.e. the probability that the frame matches the question.
- **CLIP (Q-Frame):** cosine similarity times CLIP's learned logit scale (~100), which is the inner product the
  paper's softmax expects.

In [ ]:
_scorers = {}


def _proj(out):
    # transformers v5 returns an output object from get_*_features; pooler_output holds the projected embedding
    return out if torch.is_tensor(out) else out.pooler_output


def _score_path(kind, s):
    return SCORE_DIR / kind / f"{s['qid']}.npy"


def blip_scores(s, batch=16):
    """AKS: BLIP image-text matching probability for every frame, against the question."""
    p = _score_path("blip", s)
    if p.exists():
        return np.load(p)
    if "blip" not in _scorers:
        from transformers import AutoProcessor, BlipForImageTextRetrieval
        proc = AutoProcessor.from_pretrained(BLIP_ID)
        model = BlipForImageTextRetrieval.from_pretrained(BLIP_ID, dtype=torch.float16).to("cuda").eval()
        _scorers["blip"] = (proc, model)
    proc, model = _scorers["blip"]
    out = []
    with torch.no_grad():
        for i in range(0, len(s["frames"]), batch):
            ims = [core.load_frame(f["path"], core.CLOSEUP_BOX) for f in s["frames"][i:i + batch]]
            inp = proc(images=ims, text=[s["question"]] * len(ims), truncation=True, max_length=64,
                       return_tensors="pt").to("cuda")
            inp["pixel_values"] = inp["pixel_values"].half()
            logits = model(**inp, use_itm_head=True)[0]           # (B, 2): [no match, match]
            out.append(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
    arr = np.concatenate(out).astype(np.float32)
    p.parent.mkdir(parents=True, exist_ok=True)
    np.save(p, arr)
    return arr


def clip_scores(s, batch=64):
    """Q-Frame: CLIP inner product between the question and every frame (cosine x logit scale)."""
    p = _score_path("clip", s)
    if p.exists():
        return np.load(p)
    if "clip" not in _scorers:
        from transformers import CLIPModel, CLIPProcessor
        proc = CLIPProcessor.from_pretrained(CLIP_ID)
        model = CLIPModel.from_pretrained(CLIP_ID, dtype=torch.float16).to("cuda").eval()
        _scorers["clip"] = (proc, model)
    proc, model = _scorers["clip"]
    embs = []
    with torch.no_grad():
        for i in range(0, len(s["frames"]), batch):
            ims = [core.load_frame(f["path"], core.CLOSEUP_BOX) for f in s["frames"][i:i + batch]]
            px = proc(images=ims, return_tensors="pt")["pixel_values"].half().to("cuda")
            e = _proj(model.get_image_features(pixel_values=px)).float()
            embs.append(torch.nn.functional.normalize(e, dim=-1).cpu())
        tin = proc(text=[s["question"]], return_tensors="pt", truncation=True, max_length=77, padding=True).to("cuda")
        t = torch.nn.functional.normalize(_proj(model.get_text_features(
            input_ids=tin["input_ids"], attention_mask=tin["attention_mask"])).float(), dim=-1).cpu()
        scale = model.logit_scale.exp().float().cpu().item()
    arr = (torch.cat(embs) @ t[0] * scale).numpy().astype(np.float32)
    p.parent.mkdir(parents=True, exist_ok=True)
    np.save(p, arr)
    return arr


def precompute_scores(samples):
    for s in tqdm(samples, desc="relevance scores"):
        blip_scores(s)
        clip_scores(s)
    for k in list(_scorers):                 # free the scorers before the VLM is loaded
        del _scorers[k]
    torch.cuda.empty_cache()


precompute_scores(TEST)

## 3. The selection methods

**AKS** (`frame_select.py` from the official repo): min-max normalise the relevance scores, then recursively
split the video in half while a segment looks "flat" (the mean of its top-n scores is not more than `t1` above
its own mean), up to `all_depth` levels. Each leaf segment at depth d contributes its top `num / 2^d` frames.

**Q-Frame:** `pi = softmax(I / tau)` over CLIP relevance `I`, add Gumbel noise `g = -log(-log(u))`, and keep the
top-K of `log(pi) + g` (Gumbel top-k = sampling without replacement). The multi-resolution variant then assigns
the top-ranked frames to high resolution, the next to medium, the rest to low, keeping their 1 : 2 : 8 ratio.

In [ ]:
import heapq


def aks_meanstd(dic_scores, n, fns, t1=AKS_T1, t2=AKS_T2, all_depth=AKS_DEPTH):
    """Verbatim port of meanstd() in ncTimTang/AKS frame_select.py."""
    split_scores, split_fn, no_split_scores, no_split_fn = [], [], [], []
    for dic_score, fn in zip(dic_scores, fns):
        score, depth = dic_score["score"], dic_score["depth"]
        mean, std = np.mean(score), np.std(score)
        top_n = heapq.nlargest(n, range(len(score)), score.__getitem__)
        mean_diff = np.mean([score[t] for t in top_n]) - mean
        if mean_diff > t1 and std > t2:
            no_split_scores.append(dic_score); no_split_fn.append(fn)
        elif depth < all_depth:
            half = len(score) // 2
            split_scores += [dict(score=score[:half], depth=depth + 1), dict(score=score[half:], depth=depth + 1)]
            split_fn += [fn[:half], fn[half:]]
        else:
            no_split_scores.append(dic_score); no_split_fn.append(fn)
    if split_scores:
        a, b = aks_meanstd(split_scores, n, split_fn, t1, t2, all_depth)
    else:
        a, b = [], []
    return no_split_scores + a, no_split_fn + b


def sel_aks(s, k):
    score = blip_scores(s)
    fn = list(range(len(score)))
    if len(score) < k:
        return fn
    # The repo runs with max_num_frames=64; at our smaller budgets "k / 2**depth" would round down to zero
    # frames for deep leaves, so the recursion depth is capped at log2(k) (the deepest useful level).
    depth = min(AKS_DEPTH, max(1, int(math.log2(max(2, k)))))
    norm = (score - score.min()) / (score.max() - score.min() + 1e-8)
    segs, fns = aks_meanstd([dict(score=list(norm), depth=0)], k, [fn], all_depth=depth)
    out = []
    for seg, f in zip(segs, fns):
        f_num = max(1, int(k / 2 ** seg["depth"]))
        out += [f[t] for t in heapq.nlargest(f_num, range(len(seg["score"])), seg["score"].__getitem__)]
    out = sorted(set(out))
    if len(out) > k:                                   # keep the k most relevant if the leaves overshoot
        out = sorted(heapq.nlargest(k, out, key=lambda i: norm[i]))
    for i in np.argsort(-norm):                        # top up to the exact budget for a fair comparison
        if len(out) >= k:
            break
        if int(i) not in out:
            out.append(int(i))
    return sorted(out)


def qframe_rank(s):
    """Gumbel-Max perturbed ranking of the frames, deterministic per question."""
    I = clip_scores(s)
    pi = torch.softmax(torch.from_numpy(I / QFRAME_TAU), dim=0).numpy()
    rng = np.random.default_rng(abs(hash(s["qid"])) % (2 ** 32))
    g = -np.log(-np.log(rng.uniform(1e-12, 1.0, size=len(I))))
    return np.argsort(-(np.log(pi + 1e-12) + g))          # best first


def sel_qframe(s, k):
    return sorted(int(i) for i in qframe_rank(s)[:k])


def sel_qframe_multires(s, token_budget):
    """High / medium / low tiers in Q-Frame's 1:2:8 ratio, scaled to the token budget."""
    hi, mid, lo = QFRAME_TIERS
    unit_cost = (hi * core.TOKENS_PER_FRAME[core.CLOSEUP_BOX] + mid * core.TOKENS_PER_FRAME[core.SUMMARY_BOX]
                 + lo * core.TOKENS_PER_FRAME[core.LOW_BOX])
    u = max(1, int(token_budget // unit_cost))
    order = qframe_rank(s)
    n_hi, n_mid, n_lo = hi * u, mid * u, lo * u
    out = [(int(i), core.CLOSEUP_BOX) for i in order[:n_hi]]
    out += [(int(i), core.SUMMARY_BOX) for i in order[n_hi:n_hi + n_mid]]
    out += [(int(i), core.LOW_BOX) for i in order[n_hi + n_mid:n_hi + n_mid + n_lo]]
    return sorted(out, key=lambda fb: fb[0])


def sel_bolt(s, k, alpha=BOLT_ALPHA):
    """BOLT: inverse transform sampling over sharpened relevance scores (keeps temporal order)."""
    sc = clip_scores(s).astype(np.float64)
    r = ((sc - sc.min()) / (sc.max() - sc.min() + 1e-12)) ** alpha
    cdf = np.cumsum(r) / (r.sum() + 1e-12)
    picked = []
    for i in range(1, k + 1):                       # uniform quantiles of the CDF
        j = int(np.searchsorted(cdf, i / k, side="left"))
        picked.append(min(j, len(cdf) - 1))
    out = sorted(set(picked))
    for i in np.argsort(-sc):                       # duplicates collapse; top up to the exact budget
        if len(out) >= k:
            break
        if int(i) not in out:
            out.append(int(i))
    return sorted(out)


def sel_maxinfo(s, k, pca_dim=MAXINFO_PCA_DIM):
    """MaxInfo: greedy maximum-volume subset of the frame embeddings (query-free diversity)."""
    E = core.signals(s)["emb"].astype(np.float64)
    if pca_dim and E.shape[1] > pca_dim:            # "fast" variant: reduce dimensions first
        Ec = E - E.mean(0)
        _, _, vt = np.linalg.svd(Ec, full_matrices=False)
        E = Ec @ vt[:pca_dim].T
    R, picked = E.copy(), []
    for _ in range(min(k, len(E))):                 # greedy QR-style pivoting: largest residual norm wins
        norms = np.linalg.norm(R, axis=1)
        norms[picked] = -1
        i = int(np.argmax(norms))
        picked.append(i)
        v = R[i] / (np.linalg.norm(R[i]) + 1e-12)
        R = R - np.outer(R @ v, v)
    return sorted(picked)


def sel_uniform(s, k):
    return core.uniform_idx(len(s["frames"]), k)


def ours_budget(s, token_budget):
    """Our two-tier split: cluster summary first (cheap), close-ups with what is left."""
    K = min(core.n_clusters(s), int(OURS_SUMMARY_SHARE * token_budget / core.TOKENS_PER_FRAME[core.SUMMARY_BOX]))
    K = max(1, min(K, len(s["frames"])))
    cl = core.clusters(s)
    summary = list(cl["reps"])[:K]
    left = token_budget - len(summary) * core.TOKENS_PER_FRAME[core.SUMMARY_BOX]
    C = max(1, int(left / core.TOKENS_PER_FRAME[core.CLOSEUP_BOX]))
    sig = core.signals(s)
    rel = sig["rel"] + 0.3 * sig["q"]
    avail = [i for i in range(sig["n"]) if i not in set(summary)] or list(range(sig["n"]))
    # same round-robin over clusters as notebook 05
    count = {j: 0 for j in range(len(cl["members"]))}
    size = cl["size"] / cl["size"].max()
    cscore = size * np.array([sig["rel"][m].max() for m in cl["members"]])
    picked = []
    while len(picked) < C:
        for j in sorted(count, key=lambda j: (count[j], -cscore[j])):
            cand = [i for i in cl["members"][j] if i in avail and i not in picked]
            if cand:
                picked.append(int(max(cand, key=lambda i: rel[i]))); count[j] += 1; break
        else:
            break
    return summary, sorted(picked)

## 4. Runs

In [ ]:
B = core.QwenAgent(core.QWEN_B)
print("answering model:", B.name, f"| {torch.cuda.memory_allocated() / 1e9:.1f} GB")


def run_fixed_frames(s, method, k):
    idx = {"uniform": sel_uniform, "aks": sel_aks, "qframe": sel_qframe,
           "bolt": sel_bolt, "maxinfo": sel_maxinfo}[method](s, k)
    frames = [(i, core.CLOSEUP_BOX) for i in idx]
    parts = core.frame_parts(s, frames)
    return core.answer(B, s, parts, extra=dict(method=method, budget=k, setting="fixed_frames",
                                               n_frames=len(frames), frames=idx))


def run_fixed_tokens(s, method, k):
    T = k * core.TOKENS_PER_FRAME[core.CLOSEUP_BOX]
    if method == "qframe_multires":
        frames = sel_qframe_multires(s, T)
        parts = core.frame_parts(s, frames, header=(f"The following {len(frames)} frames are sampled from the video "
                                                    "in chronological order, at mixed resolutions."))
        extra = dict(n_frames=len(frames), frames=[i for i, _ in frames],
                     tiers=[core.TOKENS_PER_FRAME[b] for _, b in frames])
    else:                                     # ours
        summary, closeups = ours_budget(s, T)
        parts = core.two_tier_parts(s, summary, closeups)
        extra = dict(n_frames=len(summary) + len(closeups), n_summary=len(summary), n_closeups=len(closeups),
                     frames=sorted(set(summary) | set(closeups)))
    return core.answer(B, s, parts, extra=dict(method=method, budget=k, setting="fixed_tokens", **extra))


RUNS = ([("fixed_frames", m, k) for k in BUDGETS for m in ["uniform", "aks", "qframe", "bolt", "maxinfo"]] +
        [("fixed_tokens", m, k) for k in BUDGETS for m in ["qframe_multires", "ours"]])
results = {}
for setting, method, k in RUNS:
    fn = (lambda s, m=method, kk=k: run_fixed_frames(s, m, kk)) if setting == "fixed_frames" else \
         (lambda s, m=method, kk=k: run_fixed_tokens(s, m, kk))
    results[(setting, method, k)] = core.run_split(fn, TEST, RESULTS_DIR / f"{B.name}__{setting}__{method}__k{k}.jsonl",
                                                   desc=f"{method} k={k}", progress=tqdm)

## 5. Results

In [ ]:
rows = []
for (setting, method, k), df in results.items():
    d = df[df.error.isna()] if "error" in df else df
    if not len(d) or "correct" not in d:
        print(f"(skipping {setting}/{method}/k={k}: no successful rows)")
        continue
    rows.append(dict(setting=setting, method=method, budget=k, n=len(d), accuracy=100 * d.correct.mean(),
                     mvbench=100 * d[d.dataset == "mvbench"].correct.mean(),
                     videomme=100 * d[d.dataset == "videomme"].correct.mean(),
                     frames=d.n_frames.mean(), tokens=d.usage.map(lambda u: u["in_tokens"]).mean(),
                     KB=d.payload_bytes.mean() / 1024, seconds=d.seconds.mean()))
table = pd.DataFrame(rows).sort_values(["budget", "setting", "method"])
print("Test split (350 questions), answering model:", B.name)
table.round(1)

In [ ]:
# Accuracy vs. input tokens: the efficiency curve
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
for ax, (name, sub) in zip(axes, [("Both datasets", table), ("MVBench", table), ("Video-MME", table)]):
    col = {"Both datasets": "accuracy", "MVBench": "mvbench", "Video-MME": "videomme"}[name]
    for method, g in sub.groupby("method"):
        g = g.sort_values("tokens")
        ax.plot(g.tokens, g[col], "o-", ms=5, label=method)
    ax.set_title(name); ax.set_xlabel("input tokens per question"); ax.grid(alpha=0.3)
axes[0].set_ylabel("accuracy (%)"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Per-category accuracy at each budget
ok_runs = {(st, m, k): df[df.error.isna()] for (st, m, k), df in results.items()
           if "correct" in df and len(df[df.error.isna()])}
acc = pd.DataFrame({f"{m}@{k}": df.set_index("qid").correct.astype(float) for (st, m, k), df in ok_runs.items()})
meta = next(iter(ok_runs.values())).set_index("qid")[["dataset", "category"]]
acc = meta.join(acc)
display((100 * acc.groupby("dataset").mean(numeric_only=True)).round(1))
display((100 * acc.groupby(["dataset", "category"]).mean(numeric_only=True)).round(1))

In [ ]:
from math import comb


def mcnemar(x, y):
    x, y = np.asarray(x, bool), np.asarray(y, bool)
    b, c = int((x & ~y).sum()), int((~x & y).sum())
    n = b + c
    return b, c, round(min(1.0, 2 * sum(comb(n, i) for i in range(min(b, c) + 1)) / 2 ** n) if n else 1.0, 3)


print("Ours vs each method, at the same budget (wins, losses, p):")
for k in BUDGETS:
    for m in ["uniform", "aks", "qframe", "bolt", "maxinfo", "qframe_multires"]:
        a, b_ = f"ours@{k}", f"{m}@{k}"
        if a in acc and b_ in acc:
            pair = acc[[a, b_]].dropna()
            print(f"  budget {k:2d}:  ours vs {m:16s} {mcnemar(pair[a], pair[b_])}")

In [ ]:
# What the methods actually pick: selected timestamps for one question
s = next(x for x in TEST if x["dataset"] == "videomme" and len(x["frames"]) > 200)
k = 16
print(f"{s['qid']} [{s['category']}] {len(s['frames'])} frames, {s['span_s']:.0f}s")
print(" ".join(s["question"].split())[:150])
for name, idx in [("uniform", sel_uniform(s, k)), ("aks", sel_aks(s, k)), ("qframe", sel_qframe(s, k)),
                  ("bolt", sel_bolt(s, k)), ("maxinfo", sel_maxinfo(s, k))]:
    print(f"  {name:8s} {[round(s['frames'][i]['t']) for i in idx]}")
summary, closeups = ours_budget(s, k * core.TOKENS_PER_FRAME[core.CLOSEUP_BOX])
print(f"  ours     summary {len(summary)} frames + close-ups {[round(s['frames'][i]['t']) for i in closeups]}")
mr = sel_qframe_multires(s, k * core.TOKENS_PER_FRAME[core.CLOSEUP_BOX])
print(f"  qframe-multires {len(mr)} frames: "
      f"{sum(1 for _, b in mr if b == core.CLOSEUP_BOX)} high, {sum(1 for _, b in mr if b == core.SUMMARY_BOX)} mid, "
      f"{sum(1 for _, b in mr if b == core.LOW_BOX)} low")

In [ ]:
B.close(); del B
import gc; gc.collect(); torch.cuda.empty_cache()